# M56 M54 FP16 parameter-storage sensitivity

This is a no-training, unchanged-graph compression experiment. It stores ordinary Conv2d/Linear parameters in FP16, reloads them into FP32 MonoDGP, and stops after a CUDA parity/profile smoke test. **Do not run the complete evaluation until the smoke report has been reviewed.**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
import json, os, shlex, shutil, subprocess, sys
MOBILE_REPO=Path('/content/mobile_adas3d')
MONODGP_REPO=Path('/content/MonoDGP_M56')
MONODGP_COMMIT='aa059a18214aebf644510e7f0793971b403f9d14'
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT=Path('/content/kitti')
SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
DATASET_ROOT=Path('/content/monodgp_kitti_m56')
M54_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/challengers/monodgp_m54')
M54_SELECTION=M54_ROOT/'product_checkpoint_sweep/m54_product_selection.json'
M55_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m55_feasibility')
M55_GATE=M55_ROOT/'m55_feasibility_gate.json'
M55_PROFILE=M55_ROOT/'m55_native_baseline_profile.json'
M55_AUDIT=M55_ROOT/'m55_operator_export_audit.json'
OUTPUT_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m56_fp16_storage')
EVALUATION_DIR=OUTPUT_ROOT/'complete_evaluation'
LOG_DIR=OUTPUT_ROOT/'colab_logs'
def run(command,cwd=None,env=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    merged=os.environ.copy(); merged.update(env or {})
    result=subprocess.run(command,cwd=cwd,env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
def run_logged(command,cwd,log_path,env=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    log_path=Path(log_path); log_path.parent.mkdir(parents=True,exist_ok=True)
    merged=os.environ.copy(); merged.update(env or {}); tail=deque(maxlen=100)
    with log_path.open('w',encoding='utf-8') as log:
        process=subprocess.Popen(command,cwd=cwd,env=merged,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True); log.write(line); tail.append(line.rstrip())
        code=process.wait()
    if code: raise RuntimeError(f'Exit {code}; full log={log_path}\n'+'\n'.join(tail))
    return log_path
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
run(['nvidia-smi'])


In [ ]:
# Fetch exact sources, apply the audited compatibility/taxonomy patches, and build the CUDA extension.
if not MOBILE_REPO.exists():
    run(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO])
else:
    run(['git','pull','--ff-only'],cwd=MOBILE_REPO)
if not MONODGP_REPO.exists():
    run(['git','clone','https://github.com/PuFanqi23/MonoDGP.git',MONODGP_REPO])
run(['git','fetch','--all'],cwd=MONODGP_REPO)
run(['git','checkout',MONODGP_COMMIT],cwd=MONODGP_REPO)
run([sys.executable,'-m','pip','install','-q','pyyaml','scipy','opencv-python-headless','numba','scikit-image','scikit-learn','tqdm','ninja','pandas'])
run([sys.executable,'scripts/patch_monodgp_colab_compat.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
run([sys.executable,'scripts/patch_monodgp_m54_training.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
changed=set(subprocess.run(['git','diff','--name-only'],cwd=MONODGP_REPO,check=True,capture_output=True,text=True).stdout.splitlines())
expected={
    'lib/datasets/kitti/kitti_dataset.py',
    'lib/helpers/save_helper.py',
    'lib/helpers/trainer_helper.py',
    'lib/models/monodgp/ops/modules/ms_deform_attn.py',
    'lib/models/monodgp/ops/setup.py',
    'lib/models/monodgp/ops/src/cuda/ms_deform_attn_cuda.cu',
    'tools/train_val.py',
}
if changed != expected: raise RuntimeError(f'Unexpected patched source set: {changed}')
ops=MONODGP_REPO/'lib/models/monodgp/ops'
shutil.rmtree(ops/'build',ignore_errors=True)
run([sys.executable,'setup.py','build','install'],cwd=ops,env={'MAX_JOBS':'2'})
run([sys.executable,'-c','import torch, MultiScaleDeformableAttention; print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0))'],cwd=MONODGP_REPO)


In [ ]:
# Create the canonical Chen-split KITTI view without copying images.
def resolve(root,names):
    for name in names:
        path=root/name
        if path.is_dir(): return path
sources={
    key:resolve(LOCAL_DATASET_ROOT,names) or resolve(DRIVE_DATASET_ROOT,names)
    for key,names in {
        'image_2':['training/image_2','training/image_02'],
        'label_2':['training/label_2','training/label_02'],
        'calib':['training/calib'],
    }.items()
}
if any(path is None for path in sources.values()): raise FileNotFoundError(sources)
(DATASET_ROOT/'training').mkdir(parents=True,exist_ok=True)
(DATASET_ROOT/'ImageSets').mkdir(parents=True,exist_ok=True)
for name,target in sources.items():
    link=DATASET_ROOT/'training'/name
    if link.is_symlink() and link.resolve()==target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target,target_is_directory=True)
for split in ('train','val'):
    shutil.copy2(SPLIT_DIR/f'{split}.txt',DATASET_ROOT/'ImageSets'/f'{split}.txt')
assert len((DATASET_ROOT/'ImageSets/train.txt').read_text().splitlines())==3712
assert len((DATASET_ROOT/'ImageSets/val.txt').read_text().splitlines())==3769
for required in (M54_SELECTION,M55_GATE,M55_PROFILE,M55_AUDIT):
    if not required.is_file(): raise FileNotFoundError(required)


In [ ]:
# Generate paired model-only FP32/FP16-storage artifacts and freeze every hash.
PREPARE_LOG=run_logged([
    sys.executable,'-u','scripts/prepare_monodgp_m56_fp16_storage.py',
    '--monodgp-repo',MONODGP_REPO,
    '--dataset-root',DATASET_ROOT,
    '--m54-selection',M54_SELECTION,
    '--m55-gate',M55_GATE,
    '--m55-profile',M55_PROFILE,
    '--m55-audit',M55_AUDIT,
    '--output-root',OUTPUT_ROOT,
],MOBILE_REPO,LOG_DIR/'m56_prepare.log')
MANIFEST=OUTPUT_ROOT/'m56_compression_manifest.json'
manifest=json.loads(MANIFEST.read_text())
assert manifest['smoke_authorized'] and all(manifest['preparation_gate_results'].values())
assert manifest['training_performed'] is False and manifest['graph_changed'] is False
assert manifest['runtime_precision']=='FP32 after checkpoint load'
print('FP32 model-only bytes:',manifest['baseline_model_only_checkpoint_bytes'])
print('FP16-storage candidate bytes:',manifest['candidate_checkpoint_bytes'])
print('Size ratio:',manifest['model_only_checkpoint_size_ratio'])
print('Candidate:',manifest['candidate_checkpoint'])


In [ ]:
# Real CUDA barrier: exact stored dtypes, noneligible parity, raw outputs, and 5/100 timing.
SMOKE=OUTPUT_ROOT/'m56_fp16_storage_smoke.json'
SMOKE_LOG=run_logged([
    sys.executable,'-u','scripts/smoke_test_monodgp_m56_fp16_storage.py',
    '--monodgp-repo',MONODGP_REPO,
    '--manifest',MANIFEST,
    '--m55-profile',M55_PROFILE,
    '--output',SMOKE,
],MOBILE_REPO,LOG_DIR/'m56_smoke.log')
smoke=json.loads(SMOKE.read_text())
assert smoke['all_smoke_gates_passed'] and smoke['full_evaluation_authorized']
print('M56 smoke gates:',json.dumps(smoke['gate_results'],indent=2))
print('Raw-output parity:',json.dumps(smoke['parity_summary'],indent=2))
print('Candidate latency:',json.dumps(smoke['latency'],indent=2))
print('M55 comparison:',json.dumps(smoke['m55_latency_comparison'],indent=2))


## Stop point 1 — return the smoke evidence

Stop here and return `m56_compression_manifest.json` and `m56_fp16_storage_smoke.json`. Do not run the complete evaluation cell below until the smoke gate is reviewed. No training has been performed.


## Complete validation — only after approval

The next cell performs one complete 3,769-image inference and evaluates all frozen AP, nearby-recall, localization, and completeness gates. It reuses only a complete prediction cache bound to the exact candidate hash.


In [ ]:
# Run only after the M56 smoke report has been reviewed and continuation is authorized.
MANIFEST=OUTPUT_ROOT/'m56_compression_manifest.json'
SMOKE=OUTPUT_ROOT/'m56_fp16_storage_smoke.json'
smoke=json.loads(SMOKE.read_text())
if not smoke.get('full_evaluation_authorized'): raise RuntimeError('M56 smoke did not authorize full evaluation')
EVAL_LOG=run_logged([
    sys.executable,'-u','scripts/evaluate_monodgp_m56_fp16_storage.py',
    '--mobile-repo',MOBILE_REPO,
    '--monodgp-repo',MONODGP_REPO,
    '--manifest',MANIFEST,
    '--smoke',SMOKE,
    '--dataset-root',DATASET_ROOT,
    '--split-dir',SPLIT_DIR,
    '--output-dir',EVALUATION_DIR,
],MOBILE_REPO,LOG_DIR/'m56_complete_evaluation.log')
GATE=EVALUATION_DIR/'m56_fp16_storage_gate.json'
gate=json.loads(GATE.read_text())
print('Candidate metrics:',json.dumps(gate['candidate_metrics'],indent=2))
print('Preservation gates:',json.dumps(gate['preservation_gate_results'],indent=2))
print('Compressed candidate selected:',gate['offline_compression_candidate_selected'])
print('Gate report:',GATE)
print('Comparison CSV:',EVALUATION_DIR/'m56_fp16_storage_comparison.csv')


## Final stop point

Return `m56_fp16_storage_gate.json` and `m56_fp16_storage_comparison.csv`. A pass selects only the offline compressed artifact and authorizes M57 operator work; it does not authorize Core ML conversion or claim product safety.
